In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split as tts

In [ ]:
df = pd.read_csv("Advertising.csv")
df.head(3)

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3


In [ ]:
df = df.drop("Unnamed: 0", axis=1)

In [ ]:
df.head(2)

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4


In [ ]:
x = df.drop("Sales", axis=1)
y = df['Sales']

print(x.shape)
print(y.shape)

(200, 3)
(200,)


In [ ]:
x_train,x_test,y_train,y_test = tts(x,y, test_size=0.2, random_state=42)

In [ ]:
lr = LinearRegression()
lr.fit(x_train,y_train)

LinearRegression()

In [ ]:
print("coeff:",lr.coef_)
print("intercept:",lr.intercept_)

coeff: [0.04472952 0.18919505 0.00276111]
intercept: 2.979067338122629


In [ ]:
y_pred = lr.predict(x_test)

from sklearn.metrics import r2_score

r2 = r2_score(y_test,y_pred)
print(r2)

0.899438024100912


<h1> Using Class

In [ ]:
x_train.shape

(160, 3)

coeff: [0.04472952 0.18919505 0.00276111]
intercept: 2.979067338122629

In [ ]:
class SGDRegressor:

  def __init__(self,learning_rate=0.01,epochs=100):
    self.learning_rate = learning_rate
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None

  def fit(self,x_train,y_train):

    # initializing random coef & intercept
    self.coef_ = np.zeros(x_train.shape[1])
    self.intercept_ = 0

    print(self.coef_,self.intercept_)

    for i in range(self.epochs):
      for j in range(x_train.shape[0]):

        idx = np.random.randint(0,x_train.shape[0])
        # vectorization
        y_hat = np.dot(x_train.iloc[idx], self.coef_) + self.intercept_

        intercept_der = -2 * (y_train.iloc[idx] - y_hat)
        self.intercept_ -= self.learning_rate * intercept_der

        coef_der = -2 * np.dot((y_train.iloc[idx] - y_hat), x_train.iloc[idx])
        self.coef_ -= self.learning_rate * coef_der

        # print control
      if i < 200 or i >= self.epochs - 200:
            print(f"epoch={i}, coef={self.coef_}, intercept={self.intercept_}")


  def predict(self,x_test):
    return np.dot(x_test,self.coef_) + self.intercept_

In [ ]:
gd = SGDRegressor(learning_rate=0.000019,epochs=90)

In [ ]:
gd.fit(x_train,y_train)

[0. 0. 0.] 0
epoch=0, coef=[0.05608545 0.11893377 0.05319536], intercept=0.003324221210687631
epoch=1, coef=[0.02004131 0.19478457 0.061454  ], intercept=0.007431357991264664
epoch=2, coef=[0.03498429 0.21154623 0.01298647], intercept=0.009595655426827051
epoch=3, coef=[0.05750352 0.19055101 0.0009114 ], intercept=0.010897722296437318
epoch=4, coef=[0.03516878 0.2251823  0.01439571], intercept=0.013455943202712537
epoch=5, coef=[0.1065213  0.21386968 0.03473104], intercept=0.016497135333771628
epoch=6, coef=[0.06081155 0.19245788 0.02172243], intercept=0.019332400005459865
epoch=7, coef=[0.06024993 0.23454467 0.00472077], intercept=0.021199708041950358
epoch=8, coef=[ 0.05685931  0.26407003 -0.00413571], intercept=0.023387766004623528
epoch=9, coef=[ 0.05262555  0.24531214 -0.00137911], intercept=0.02665712827076665
epoch=10, coef=[ 0.04323917  0.23894997 -0.00763448], intercept=0.031091729334909588
epoch=11, coef=[0.04791486 0.22842062 0.023911  ], intercept=0.03335050795618497
epoch=

In [ ]:
y_pred = gd.predict(x_test)

In [ ]:
r2 = r2_score(y_test, y_pred)

In [ ]:
r2

0.8654905085336222

Our r2 score always change's because it takes only one random row not the whole dataset. whenever row change's score also changes.

got r2 0.8904600352844214 in batch gd

<h1> Scaled values

In [ ]:
from math import inf
x_scaled = (x - x.mean()) / x.std()
x_train,x_test,y_train,y_test = tts(x_scaled,y, test_size=0.2, random_state=42)



class SGDRegressor:

  def __init__(self,learning_rate=0.01,epochs=100, tolerance = 1e-17, patience = 15):
    self.learning_rate = learning_rate
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None
    self.tolerance = tolerance
    self.patience = patience

  def fit(self,x_train,y_train):

    # initializing random coef & intercept
    self.coef_ = np.ones(x_train.shape[1])
    self.intercept_ = 0

    previous_loss = float(inf)
    patience_counter = 0

    print(self.coef_,self.intercept_)

    for i in range(self.epochs):
      for j in range(x_train.shape[0]):

        idx = np.random.randint(0,x_train.shape[0])

        y_hat = np.dot(x_train.iloc[idx], self.coef_) + self.intercept_

        # Loss (MSE)
        loss = np.mean((y_train.iloc[idx] - y_hat) ** 2)

        intercept_der = -2 * (y_train.iloc[idx] - y_hat)
        self.intercept_ -= self.learning_rate * intercept_der

        coef_der = -2 * np.dot((y_train.iloc[idx] - y_hat), x_train.iloc[idx])
        self.coef_ -= self.learning_rate * coef_der

        # print control
      if i < 200 or i >= self.epochs - 200:
            print(f"epoch={i}, coef={self.coef_}, intercept={self.intercept_}")


        # early stopping
      if abs(previous_loss - loss) < self.tolerance:
        patience_counter +=1
        if patience_counter >= self.patience:
          print(f"Early stopping at epochs {i}")
          break
      else:
        patience_counter = 0
        previous_loss = loss

  def predict(self,x_test):
    return np.dot(x_test,self.coef_) + self.intercept_

In [ ]:
x_train.head(2)

,TV,Radio,Newspaper
79,-0.361572,-1.048306,-0.342262
197,0.348934,-0.940539,-1.109069


In [ ]:
gd = SGDRegressor(epochs=100,learning_rate=0.001)

In [ ]:
gd.fit(x_train,y_train)

[1. 1. 1.] 0
epoch=0, coef=[1.99170954 2.09704688 0.9609661 ], intercept=3.8594129079805235
epoch=1, coef=[2.7437972  2.04580636 0.61204859], intercept=6.638075516173456
epoch=2, coef=[2.98622654 2.10511127 0.54705284], intercept=8.686730090624494
epoch=3, coef=[3.3396795  2.2586462  0.52541885], intercept=10.193300645817061
epoch=4, coef=[3.58795458 2.42177043 0.39737575], intercept=11.254723959535214
epoch=5, coef=[3.66919505 2.43759423 0.25718988], intercept=12.061605085683377
epoch=6, coef=[3.76334962 2.57849584 0.29674045], intercept=12.629118302817433
epoch=7, coef=[3.77869007 2.63047969 0.20748411], intercept=13.02540868284302
epoch=8, coef=[3.85903222 2.64937645 0.21452195], intercept=13.241410935752542
epoch=9, coef=[3.89419727 2.68963678 0.21639512], intercept=13.473647534721168
epoch=10, coef=[3.89328217 2.60882016 0.22879265], intercept=13.603175035467979
epoch=11, coef=[3.88349813 2.63017043 0.23780758], intercept=13.78537966855387
epoch=12, coef=[3.95633057 2.68245636 0.2

In [ ]:
y_pred = gd.predict(x_test)

In [ ]:
r2 = r2_score(y_test, y_pred)
r2

0.898298971651692

In [ ]:
sk_pred = 0.899438024100912
gd_pred = r2

if gd_pred > sk_pred:
  diff_gd = gd_pred - sk_pred
  print("GD working better:",diff_gd)
elif gd_pred < sk_pred:
  diff_sk = sk_pred - gd_pred
  print("sklearn model working better with",diff_sk)
else:
  print("No difference both got same score")

sklearn model working better with 0.0011390524492199683


To control this learning rate we apply learning shedule functions. These functions decrease learning rate over the iteration so we will not deviate from the actual minima.